# Appendix — Agno: the batteries-included agent

Seven framework appendices run the *same* Larkspur triage — ticket `TKT-2205`, gold `partial_refund | pol-restocking | $170.99` — so you can compare frameworks on one fixed problem. This is the **batteries-included** archetype: [Agno](https://docs.agno.com/) is an SDK to "build agents, teams, and workflows" (with AgentOS to run them in production). Where earlier chapters handed you a bare `run_agent` loop and a message list to manage yourself, Agno's `Agent` arrives with the loop, a tool registry, session memory, a database layer, and a human-in-the-loop confirmation gate already assembled. This appendix wires the ops desk with that machinery and points each piece back at the chapter where you built it by hand.

> **Before running this notebook:** `pip install -e ".[agno]"` (once). It pulls in `agno` (>=2.9,<3), which co-installs with the main venv — no separate kernel. The model reaches the same OpenRouter endpoint you have used since ch01, through Agno's native `OpenRouter` model class. Everything else stays the same.

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

## The model boundary: a native OpenRouter class

Agno reaches the model through one of its built-in model classes, not `shoplab.llm.complete` or LiteLLM. It ships `agno.models.openrouter.OpenRouter`, which already points at OpenRouter's OpenAI-compatible endpoint (`https://openrouter.ai/api/v1`), so the model id is just the raw `deepseek/deepseek-v3.2` — no `openrouter/` routing prefix (that was LiteLLM's syntax). We set `temperature=0` to keep the course's determinism. One consequence worth repeating from ch16: these calls go through Agno's own OpenAI client, so they skip the LiteLLM disk cache — a rerun costs the same cent or two.

In [ ]:
import os
from agno.models.openrouter import OpenRouter

model = OpenRouter(id="deepseek/deepseek-v3.2",
                   api_key=os.environ["OPENROUTER_API_KEY"],
                   temperature=TEMPERATURE,        # the whole course runs at temp 0
                   max_tokens=2048)
print("model:", type(model).__module__ + "." + type(model).__name__)
print("id:", model.id, "| base_url:", model.base_url)

## The invariant: gold from `shoplab.rules.decide`

Every appendix must land on the same economics, so pin the invariant now — the authoritative gold from the deterministic rules engine. `TKT-2205` is an opened, in-window return from a non-vip member, so the cascade applies a 10% restocking fee (`$189.99 x 0.90`) and lands on `partial_refund | pol-restocking | 170.99`. That triple is what the agent must reproduce.

In [ ]:
from shoplab import world, rules

orders = {o["order_id"]: o for o in world.load_orders()}
customers = {c["customer_id"]: c for c in world.load_customers()}
t2205 = next(t for t in world.load_tickets()["train"] if t["ticket_id"] == "TKT-2205")
GOLD = rules.decide(t2205, orders[t2205["order_id"]], customers[t2205["customer_id"]])
print("authoritative gold (shoplab.rules.decide):", GOLD)   # the invariant every appendix hits

## Tools: plain Python functions

Agno takes tools as plain annotated Python functions — the docstring becomes the description, the type hints become the JSON schema — exactly what `shoplab.tools.to_openai_tools` did by hand in chapter 02. The four read-only lookups (`get_order`, `get_customer`, `search_policy`, `calc`) are thin wrappers that *import* from `shoplab.world` / `shoplab.tools`, not reimplementations; `calc` catches the strict engine's `ValueError` and hands it back as text, mirroring `run_tool`'s "never raise" contract. The risky `issue_refund` is decorated `@tool(requires_confirmation=True)` — the one flag that turns it into the approval gate of the next section.

In [ ]:
from agno.agent import Agent
from agno.tools import tool
from shoplab.tools import calc as _calc, Ledger

ledger = Ledger()                               # ch02's real side-effect log

def get_order(order_id: str) -> dict:
    "Look up a Larkspur order by id (items, totals, status, dates)."
    return orders.get(order_id, {"error": f"no such order {order_id}"})

def get_customer(customer_id: str) -> dict:
    "Look up a Larkspur customer by id (tier, flags, history)."
    return customers.get(customer_id, {"error": f"no such customer {customer_id}"})

def search_policy(query: str, k: int = 2) -> list:
    "Keyword-search the 12 Larkspur store policy documents."
    return world.search_policy(query, k=k)

def calc(expr: str) -> str:
    "Evaluate an arithmetic expression, e.g. '0.9 * 189.99'."
    try:
        return str(_calc(expr))                 # thin wrapper over shoplab.tools.calc
    except ValueError as e:                      # never raise -- like shoplab.tools.run_tool
        return f"error: {e}"

@tool(requires_confirmation=True)                # the ch08 gate, one flag
def issue_refund(order_id: str, amount_usd: float, reason: str) -> dict:
    "Send money back to the customer. Irreversible."
    amount = round(float(amount_usd), 2)         # money moves in whole cents (shoplab.rules rounds too)
    entry = ledger.record("issue_refund", order_id=order_id,
                          amount_usd=amount, reason=reason)
    return {"ok": True, "refund_id": f"REF-{1000 + len(ledger.entries)}", **entry}

INSTRUCTIONS = (
    "You are the Larkspur ops desk triaging one return ticket. Work in this exact order "
    "and do not skip a step: (1) call get_order, (2) call get_customer, (3) call "
    "search_policy for the governing rule, (4) call calc for the refund amount, (5) call "
    "issue_refund to move that exact amount. Opened non-vip returns carry a 10% restocking "
    "fee, so the refund is 90% of the item's value, rounded to the nearest cent. You must "
    "call issue_refund before you "
    "write your decision. Only after issue_refund has run, end your reply with exactly one "
    "line, no markdown: DECISION: <decision> | <policy_id> | <refund_usd>  where <decision> "
    "is one of approve_refund, partial_refund, replacement, store_credit, deny, escalate. A "
    "full refund reduced by a restocking fee is partial_refund, not approve_refund.")

agent = Agent(model=model,
              tools=[get_order, get_customer, search_policy, calc, issue_refund],
              instructions=INSTRUCTIONS)
print("wired: get_order, get_customer, search_policy, calc (read) "
      "+ issue_refund (risky, confirmation-gated)")

## The approval gate is a run that pauses

Chapter 08 wrapped the risky tools in `require_approval` so money moved only after a human said yes. Agno ships that as one flag. Because `issue_refund` is `requires_confirmation=True`, when the model calls it the run does **not** execute it — `agent.run` returns with `run.is_paused` true and the pending call sitting in `run.tools_requiring_confirmation`. Nothing has hit the `Ledger` yet. Run the triage and watch it halt in front of the money move.

In [ ]:
TICKET = ("Triage Larkspur ticket TKT-2205 (order ORD-7312, customer CUST-07, sku LK-1016, "
          "qty 1): 'I opened the box and used the Torrent boots one evening indoors, they "
          "pinch at the toes. Repacked with tags. Refund my original payment method.'")

paused = agent.run(TICKET)                       # multi-step loop, then the gate
print("is_paused:", paused.is_paused)
for call in (paused.tools_requiring_confirmation or []):
    print("pending  :", call.tool_name, call.tool_args)
print("ledger   :", len(ledger.entries), "(gate held -- no money moved yet)")

> **What you should see:** the run stops with `is_paused` true, and the pending call is `issue_refund` on `ORD-7312` for about `170.99` (the model's `189.99 x 0.90`). The `Ledger` still reads `0` — `requires_confirmation` turned the money move into a pause, not a side effect. This is chapter 08's approval gate, handed to you by one decorator flag.

In [ ]:
import re

if paused.is_paused:
    for call in paused.tools_requiring_confirmation:   # a human approves (ch08 approver -> True)
        call.confirmed = True
    done = agent.continue_run(run_response=paused)     # gate lifts, issue_refund runs
else:
    done = paused                                       # model answered without the money move

# The verdict's economics come from the money that ACTUALLY moved (the ledger),
# not the printed DECISION line -- the regex only reads back decision + policy.
moved = ledger.entries[-1]["amount_usd"] if ledger.entries else None
m = re.search(r"DECISION:\s*([a-z_]+)\s*\|\s*(pol-[a-z-]+)", done.content)
verdict = {"decision": m.group(1), "policy_id": m.group(2), "refund_usd": moved}
print("verdict       :", verdict)
print("matches gold  :", verdict == GOLD)              # the dollars moved, not just the line
print("ledger entries:", len(ledger.entries))
if ledger.entries:
    print("  last refund :", ledger.entries[-1])
print("session msgs  :", len(done.messages or []), "(agno threaded the context -- ch10)")

> **What you should see:** setting `confirmed = True` and calling `continue_run` lifts the gate -- `issue_refund` runs and the `Ledger` ticks to `1`. The verdict's `refund_usd` is read straight from that ledger entry -- the dollars that actually moved, rounded to whole cents -- and it equals the gold from `shoplab.rules.decide` (`matches gold: True`), the `$189.99 x 0.90` invariant every appendix in this set shares. The model's final line echoes the same `partial_refund | pol-restocking` decision. (Agno's structured `output_schema` would hand back a typed verdict object, but in 2.9 it short-circuits the tool loop -- the model answers without ever calling the gated tool -- so here the decision comes back as a line we parse: the chapter 01 boundary, resurfacing.)

## Machinery map: Agno feature to the part you built

Line them up and the framework stops being magic. Every Agno concept in this appendix maps to a piece of machinery you built by hand in Parts 1-3 — usually a batteries-included feature standing in for a few lines of your own code.

| Agno concept | Your hand-built equivalent | Built in |
|---|---|---|
| `Agent(model, tools, instructions)` and its `.run()` loop | `run_agent`'s while-loop: model call -> tool calls -> results, repeat | ch02 |
| plain functions in `tools=[...]` (docstring -> desc, hints -> schema) | the `Tool` registry + `to_openai_tools` building the call schema by hand | ch02 |
| `OpenRouter(id=...)` model class | `shoplab.llm.complete` wrapping the model boundary at OpenRouter | ch01 |
| `@tool(requires_confirmation=True)` + `run.is_paused` | `require_approval` wrapping a risky tool in a yes/no gate | ch08 |
| `call.confirmed = True` + `agent.continue_run(...)` | the approver returning `True` and the loop resuming from a saved checkpoint | ch08 / ch12 |
| the session message list agno threads (`run.messages`, `get_chat_history`) | the `messages` list threaded through the loop by hand | ch02 |
| `session_state` / `db` / `memory_manager` (persisted history, user memories) | `compact` / `offload` / `run_subagent` — the context-window tools | ch10 |

## The honest trade

Agno's bet is batteries-included. You wired an `Agent`, five plain functions, and one decorator flag, and got back a multi-step tool loop, a human-in-the-loop pause that stops exactly in front of the money move, and a managed session context — the ch02 loop, the ch08 gate, and the ch10 message threading, all assembled and maintained for you. For a production desk that needs persistence, memory, and approvals out of the box, that is a real buy, and nothing you learned is wasted: you can reason about each feature precisely *because* you built it once.

The cost is the usual framework tax, in a maximalist accent. The `Agent` is a large object — over a hundred constructor parameters — and most of its surface (teams, workflows, knowledge, AgentOS) is invisible on a one-ticket triage, so you pay in concepts you must learn to hold. The typed path fights the tool path: `output_schema` would give a validated verdict, but in 2.9 it short-circuits the loop, so to keep the approval gate you fall back to a text `DECISION` line and a regex — the very chapter 01 parsing you thought you had escaped. Model variance is real too: the decision vocabulary only lands reliably because the instructions pin it hard. And the seam moved, exactly as chapters 11 and 16 warned: these calls go through Agno's own OpenAI client, not `shoplab.llm.complete` or LiteLLM, so the cost `LEDGER` and the disk cache from earlier chapters no longer see them. You are buying an assembled runtime; know which seams you hand over to get it.